In [ ]:
# Necessary Imports
import os
import cv2
import torch
from ultralytics import YOLO
from tqdm import tqdm
import pretrainedmodels
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast, GradScaler  # Using GradScaler because I was running into CUDA OOM error
from PIL import Image

In [ ]:
# YOLO Pipeline for generating cropped images of leaves
RAW_DATA_DIR = "Merged_Dataset(Plant_Doc+Plant_Wild)/train"          # Path to your original folders
PROCESSED_DATA_DIR = "Merged_Dataset(Plant_Doc+Plant_Wild)/cropped_training" # Where the new dataset will go
YOLO_MODEL = YOLO("yolo_v11_plant_doc.pt")        # Use your trained YOLO
IMG_SIZE = 224                         # Final size for EfficientNet

def prepare_training_data():
    if not os.path.exists(PROCESSED_DATA_DIR):
        os.makedirs(PROCESSED_DATA_DIR)

    classes = os.listdir(RAW_DATA_DIR)
    
    for cls in classes:
        input_class_path = os.path.join(RAW_DATA_DIR, cls)
        output_class_path = os.path.join(PROCESSED_DATA_DIR, cls)
        
        if not os.path.isdir(input_class_path): continue
        os.makedirs(output_class_path, exist_ok=True)
        
        print(f"Processing Class: {cls}")
        for img_name in tqdm(os.listdir(input_class_path)):
            img_path = os.path.join(input_class_path, img_name)
            img = cv2.imread(img_path)
            if img is None: continue
            
            # Run YOLO
            results = YOLO_MODEL(img, conf=0.3, verbose=False)
            boxes = results[0].boxes
            
            # Logic: Crop if found, else keep full
            if len(boxes) > 0:
                best_box = boxes[0].xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = map(int, best_box)
                processed_img = img[y1:y2, x1:x2]
            else:
                processed_img = img
            
            try:
                processed_img = cv2.resize(processed_img, (IMG_SIZE, IMG_SIZE))
                cv2.imwrite(os.path.join(output_class_path, img_name), processed_img)
            except:
                continue

prepare_training_data()

Processing Class: Potato_Early_blight


100%|██████████| 2274/2274 [00:45<00:00, 49.78it/s]


Processing Class: Potato_healthy


100%|██████████| 2073/2073 [00:41<00:00, 50.38it/s]


Processing Class: Potato_Lateblight


...

100%|██████████| 2023/2023 [00:42<00:00, 47.47it/s]


Processing Class: Tomato_Septoria_leaf_spot


100%|██████████| 2105/2105 [00:47<00:00, 44.69it/s]


Processing Class: Tomato_Tomato_Yellow_Leaf_Curl_Virus


100%|██████████| 2202/2202 [00:47<00:00, 46.02it/s]


In [ ]:
# --- HYPERPARAMETERS ---
BATCH_SIZE = 16
EPOCHS = 20
LEARNING_RATE = 0.0001
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_transforms = {
    'train': transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# Creating the dataset and loaders
full_dataset = datasets.ImageFolder("Merged_Dataset(Plant_Doc+Plant_Wild)/cropped_training", transform=data_transforms['train'])
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_set, val_set = torch.utils.data.random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)

# Setting up the model and modifying it's classification head
model = models.efficientnet_b7(weights="DEFAULT")
# model = models.efficientnet_b4(weights="DEFAULT")
num_classes = len(full_dataset.classes)
# model.classifier[1] = nn.Linear(1792, num_classes)
model.classifier[1] = nn.Linear(2560, num_classes)
model.to(DEVICE)

# Setting up the loss function, the optimizer, and scaler
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scaler = GradScaler() # Using GradScaler because I am using an RTX GPU for training, 
                      # might need to change this code accordingly.




C:\Users\Anuraag Shukla\AppData\Local\Temp\ipykernel_23860\1505120343.py:39: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() # Using GradScaler because I am using an RTX GPU for training,


In [278]:
# For changing the learning rate after a set epochs of training
for g in optimizer.param_groups:
    g['lr'] = 0.00005
print("Learning rate adjusted for fine-tuning.")
EPOCHS = 20

Learning rate adjusted for fine-tuning.


In [279]:
# Training Loop with tqdm
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{EPOCHS}]", leave=True)
    
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        # --- MIXED PRECISION FORWARD PASS ---
        with autocast():
            outputs = model(inputs)
            loss = criterion(outputs, labels)
        
        # --- BACKWARD PASS WITH SCALER ---
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item()
        
        loop.set_postfix(loss=loss.item())

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1} Completed. Average Loss: {avg_loss:.4f}")

Epoch [1/20]:   0%|          | 0/179 [00:00<?, ?it/s]C:\Users\Anuraag Shukla\AppData\Local\Temp\ipykernel_23860\415679715.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch [1/20]: 100%|██████████| 179/179 [04:29<00:00,  1.51s/it, loss=1.78]   


Epoch 1 Completed. Average Loss: 0.1269


Epoch [2/20]: 100%|██████████| 179/179 [03:42<00:00,  1.24s/it, loss=2.11]  


Epoch 2 Completed. Average Loss: 0.1187


Epoch [3/20]: 100%|██████████| 179/179 [03:40<00:00,  1.23s/it, loss=1.53]   


...

Epoch 18 Completed. Average Loss: 0.1028


Epoch [19/20]: 100%|██████████| 179/179 [03:27<00:00,  1.16s/it, loss=2.73]   


Epoch 19 Completed. Average Loss: 0.1010


Epoch [20/20]: 100%|██████████| 179/179 [03:57<00:00,  1.33s/it, loss=3.21]    

Epoch 20 Completed. Average Loss: 0.1000


In [280]:
# Validation Block

def validate_model(model, loader, device):
    model.eval()
    correct_1 = 0
    correct_5 = 0
    total = 0
    all_preds = []
    all_labels = []

    print("Running Validation...")
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            
            # Top-1 Accuracy
            _, pred = torch.max(outputs, 1)
            total += labels.size(0)
            correct_1 += (pred == labels).sum().item()

            # Top-5 Accuracy
            _, top5_preds = outputs.topk(5, 1, True, True)
            correct_5 += top5_preds.eq(labels.view(-1, 1).expand_as(top5_preds)).sum().item()

            # Store for Report
            all_preds.extend(pred.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    top1_acc = 100 * correct_1 / total
    top5_acc = 100 * correct_5 / total

    print(f"\n--- Validation Results ---")
    print(f"Top-1 Accuracy: {top1_acc:.2f}%")
    print(f"Top-5 Accuracy: {top5_acc:.2f}%")
    
    return all_labels, all_preds

# Run the validation
y_true, y_pred = validate_model(model, val_loader, DEVICE)

Running Validation...

--- Validation Results ---
Top-1 Accuracy: 57.64%
Top-5 Accuracy: 88.50%


In [281]:
full_dataset.classes

['Potato_Early_blight',
 'Potato_Lateblight',
 'Potato_healthy',
 'Tomato_Bacterial_spot',
 'Tomato_Early_blight',
 'Tomato_Late_blight',
 'Tomato_Leaf_mold',
 'Tomato_Septoria_leaf_spot',
 'Tomato_Tomato_Yellow_Leaf_Curl_Virus',
 'Tomato_healthy',
 'Tomato_mosaic_virus']

In [ ]:
# # Run this cell to load the trained model and carry out testing
# model = torch.load("Efficient_B4_Merged_Plant_Doc_Wild_Only.pth",weights_only=False)

In [ ]:
# Testing Block

labels = full_dataset.classes

preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Merged_Dataset(Plant_Doc+Plant_Wild)/cropped_testing" 

def final_test(model, test_path):
    model.eval()
    correct = 0
    total = 0
    
    results_summary = {}

    print("Running validation on subset...")
    
    for class_folder in os.listdir(test_path):
        folder_path = os.path.join(test_path, class_folder)
        if not os.path.isdir(folder_path): continue
        
        results_summary[class_folder] = {"correct": 0, "total": 0}
        
        for img_name in tqdm(os.listdir(folder_path), desc=f"Testing {class_folder}"):

            img_path = os.path.join(folder_path, img_name)
            
            try:
                img = Image.open(img_path).convert('RGB')
            except:
                continue
                
            input_tensor = preprocess(img).unsqueeze(0).to(DEVICE)
            
            with torch.no_grad():
                output = model(input_tensor)
                _, pred_idx = torch.max(output, 1)
                
                predicted_name = labels[pred_idx.item()]
            
            if predicted_name.strip() == class_folder.strip():
                correct += 1
                results_summary[class_folder]["correct"] += 1
            
            total += 1
            results_summary[class_folder]["total"] += 1
    print(results_summary)

    print("\n" + "="*40)
    print(f"{'Class Name':<30} | {'Accuracy':<10}")
    print("-"*45)
    for cls, stats in results_summary.items():
        acc = (stats['correct']/stats['total'])*100 if stats['total'] > 0 else 0
        print(f"{cls:<30} | {acc:>8.2f}%")
    
    final_score = (correct / total) * 100
    print("="*40)
    print(f"OVERALL ACCURACY ON SUBSET: {final_score:.2f}%")

final_test(model, TEST_DIR)

Running validation on subset...


Testing Tomato_Tomato_Yellow_Leaf_Curl_Virus: 100%|██████████| 84/84 [00:04<00:00, 18.53it/s]

{'Potato_Early_blight': {'correct': 42, 'total': 87}, 'Potato_healthy': {'correct': 48, 'total': 63}, 'Potato_Lateblight': {'correct': 51, 'total': 104}, 'Tomato_Bacterial_spot': {'correct': 50, 'total': 100}, 'Tomato_Early_blight': {'correct': 29, 'total': 62}, 'Tomato_healthy': {'correct': 28, 'total': 71}, 'Tomato_Late_blight': {'correct': 41, 'total': 90}, 'Tomato_Leaf_mold': {'correct': 51, 'total': 73}, 'Tomato_mosaic_virus': {'correct': 7, 'total': 20}, 'Tomato_Septoria_leaf_spot': {'correct': 52, 'total': 108}, 'Tomato_Tomato_Yellow_Leaf_Curl_Virus': {'correct': 53, 'total': 84}}

Class Name                     | Accuracy  
---------------------------------------------
Potato_Early_blight            |    48.28%
Potato_healthy                 |    76.19%
Potato_Lateblight              |    49.04%
Tomato_Bacterial_spot          |    50.00%
Tomato_Early_blight            |    46.77%
Tomato_healthy                 |    39.44%
Tomato_Late_blight             |    45.56%
Tomato_Leaf_m

In [284]:
# Best Model as of now
model = torch.load("Efficient_B4_Merged_Plant_Doc_Wild_Only.pth", weights_only=False)
final_test(model, TEST_DIR)

Running validation on subset...


Testing Tomato_Tomato_Yellow_Leaf_Curl_Virus: 100%|██████████| 84/84 [00:02<00:00, 35.45it/s]

{'Potato_Early_blight': {'correct': 49, 'total': 87}, 'Potato_healthy': {'correct': 48, 'total': 63}, 'Potato_Lateblight': {'correct': 51, 'total': 104}, 'Tomato_Bacterial_spot': {'correct': 62, 'total': 100}, 'Tomato_Early_blight': {'correct': 35, 'total': 62}, 'Tomato_healthy': {'correct': 23, 'total': 71}, 'Tomato_Late_blight': {'correct': 45, 'total': 90}, 'Tomato_Leaf_mold': {'correct': 53, 'total': 73}, 'Tomato_mosaic_virus': {'correct': 11, 'total': 20}, 'Tomato_Septoria_leaf_spot': {'correct': 46, 'total': 108}, 'Tomato_Tomato_Yellow_Leaf_Curl_Virus': {'correct': 56, 'total': 84}}

Class Name                     | Accuracy  
---------------------------------------------
Potato_Early_blight            |    56.32%
Potato_healthy                 |    76.19%
Potato_Lateblight              |    49.04%
Tomato_Bacterial_spot          |    62.00%
Tomato_Early_blight            |    56.45%
Tomato_healthy                 |    32.39%
Tomato_Late_blight             |    50.00%
Tomato_Leaf_